In [34]:
import pandas as pd

dataset = pd.read_csv("../data/wiki_movie_plots_deduped.csv")
dataset.head()


,Release Year,Title,Origin/Ethnicity,Director,Cast,Genre,Wiki Page,Plot
0,1901,Kansas Saloon Smashers,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Kansas_Saloon_Sm...,"A bartender is working at a saloon, serving dr..."
1,1901,Love by the Light of the Moon,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/Love_by_the_Ligh...,"The moon, painted with a smiling face hangs ov..."
2,1901,The Martyred Presidents,American,Unknown,NaN,unknown,https://en.wikipedia.org/wiki/The_Martyred_Pre...,"The film, just over a minute long, is composed..."
3,1901,"Terrible Teddy, the Grizzly King",American,Unknown,NaN,unknown,"https://en.wikipedia.org/wiki/Terrible_Teddy,_...",Lasting just 61 seconds and consisting of two ...
4,1902,Jack and the Beanstalk,American,"George S. Fleming, Edwin S. Porter",NaN,unknown,https://en.wikipedia.org/wiki/Jack_and_the_Bea...,The earliest known adaptation of the classic f...


In [35]:
df = dataset[["Title", "Plot"]]
df.head()

,Title,Plot
0,Kansas Saloon Smashers,"A bartender is working at a saloon, serving dr..."
1,Love by the Light of the Moon,"The moon, painted with a smiling face hangs ov..."
2,The Martyred Presidents,"The film, just over a minute long, is composed..."
3,"Terrible Teddy, the Grizzly King",Lasting just 61 seconds and consisting of two ...
4,Jack and the Beanstalk,The earliest known adaptation of the classic f...


In [36]:
df = df[:500]
df.describe()

,Title,Plot
count,500,500
unique,488,497
top,Dr. Jekyll and Mr. Hyde,"As described in a film magazine,[1] Jules Lene..."
freq,3,2


In [37]:
df.isnull().sum()

Title    0
Plot     0
dtype: int64

In [38]:
df.duplicated().sum()

np.int64(3)

In [39]:
df = df.drop_duplicates()

In [40]:
df['Title'] = df['Title'].astype(str)
df['Plot'] = df['Plot'].astype(str)



In [41]:
df.head()

,Title,Plot
0,Kansas Saloon Smashers,"A bartender is working at a saloon, serving dr..."
1,Love by the Light of the Moon,"The moon, painted with a smiling face hangs ov..."
2,The Martyred Presidents,"The film, just over a minute long, is composed..."
3,"Terrible Teddy, the Grizzly King",Lasting just 61 seconds and consisting of two ...
4,Jack and the Beanstalk,The earliest known adaptation of the classic f...


In [42]:
def chunk_text_with_overlap(text, chunk_size=300, overlap=60):
    words = text.split()
    chunks = []

    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)

        start += (chunk_size - overlap)

    return chunks


In [43]:
documents = []

for idx, row in df.iterrows():
    title = row["Title"]
    plot = row["Plot"]

    chunks = chunk_text_with_overlap(plot, chunk_size=300, overlap=60)

    for chunk_id, chunk in enumerate(chunks):
        documents.append({
            "id": f"{idx}_{chunk_id}",
            "title": title,
            "text": chunk
        })
documents_df = pd.DataFrame(documents)
documents_df.head()

,id,title,text
0,0_0,Kansas Saloon Smashers,"A bartender is working at a saloon, serving dr..."
1,1_0,Love by the Light of the Moon,"The moon, painted with a smiling face hangs ov..."
2,2_0,The Martyred Presidents,"The film, just over a minute long, is composed..."
3,3_0,"Terrible Teddy, the Grizzly King",Lasting just 61 seconds and consisting of two ...
4,4_0,Jack and the Beanstalk,The earliest known adaptation of the classic f...


In [44]:
documents[:3]

[{'id': '0_0',
  'title': 'Kansas Saloon Smashers',
  'text': "A bartender is working at a saloon, serving drinks to customers. After he fills a stereotypically Irish man's bucket with beer, Carrie Nation and her followers burst inside. They assault the Irish man, pulling his hat over his eyes and then dumping the beer over his head. The group then begin wrecking the bar, smashing the fixtures, mirrors, and breaking the cash register. The bartender then sprays seltzer water in Nation's face before a group of policemen appear and order everybody to leave.[1]"},
 {'id': '1_0',
  'title': 'Love by the Light of the Moon',
  'text': "The moon, painted with a smiling face hangs over a park at night. A young couple walking past a fence learn on a railing and look up. The moon smiles. They embrace, and the moon's smile gets bigger. They then sit down on a bench by a tree. The moon's view is blocked, causing him to frown. In the last scene, the man fans the woman with his hat because the moon h

In [ ]:
#checking chunk overlap is working
long_text = "word " * 800  
chunks = chunk_text_with_overlap(long_text, 300, 60)

for i, chunk in enumerate(chunks):
    print(i, len(chunk.split()))


0 300
1 300
2 300
3 80


In [46]:
chunk0_end = chunks[0].split()[-60:]
chunk1_start = chunks[1].split()[:60]

print(chunk0_end == chunk1_start)


True


In [50]:
from sentence_transformers import SentenceTransformer
import chromadb

In [ ]:
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

#Chroma db client initialization
client = chromadb.Client()

#create collection
collection = client.create_collection(name="movie_plots")

for doc in documents:
    embedding = model.encode(doc['text']).tolist()
    collection.add(
        documents=[doc['text']],
        metadatas=[{"id": doc['id'], "title": doc['title']}],
        ids=[doc['id']],
        embeddings=[embedding]
    )
    

c:\Users\Hasaranga\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Hasaranga\.cache\huggingface\hub\models--sentence-transformers--all-mpnet-base-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is 

In [ ]:
import chromadb
from sentence_transformers import SentenceTransformer

# hf embedding model
model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

# use PersistentClient instead of in-memory client
client = chromadb.PersistentClient(path="../chroma_db")

collection = client.create_collection(name="movie_plots")



In [ ]:
for doc in documents: 
    embedding = model.encode(doc['text']).tolist()
    collection.add(
        documents=[doc['text']],
        metadatas=[{"id": doc['id'], "title": doc['title']}],
        ids=[doc['id']],
        embeddings=[embedding]
    )


In [4]:
import chromadb
client = chromadb.PersistentClient(path="../chroma_db")
collection = client.get_collection(name="movie_plots")


In [5]:
query = "A movie about a robot bartender"
query_embedding = model.encode(query).tolist()

# hybrid search (vector + metadata filter)
results = collection.query(
    query_embeddings=[query_embedding],
    n_results=5,
    where={"title": {"$eq": "Love by the Light of the Moon"}}  
)

print(results)


NameError: name 'model' is not defined